<a href="https://colab.research.google.com/github/ShashankMelkote11/instagram-engagement-analytics/blob/main/influencer-analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Influencer Engagement Intelligence & Behavioral Analytics

## Project Overview
This project analyzes Instagram creator performance using behavioral engagement metrics, statistical analysis, and machine learning techniques. The objective is to understand how different forms of engagement, content characteristics, posting behavior, and creator attributes influence overall post performance and virality.

## Key Objectives
- Analyze affective, cognitive, and behavioral engagement
- Engineer engagement intelligence metrics
- Segment creators by audience scale
- Identify high-performing content patterns
- Evaluate behavioral factors affecting virality
- Build predictive models for post performance

## Tools & Technologies
- Python
- Pandas & NumPy
- Matplotlib & Seaborn
- Scikit-learn
- Statistical Testing (Kruskal-Wallis, Mann-Whitney U)



In [1]:
import pandas as pd
import numpy  as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
pd.set_option('display.width', 1000)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from sklearn.preprocessing import MinMaxScaler

##SECTION 1 — DATA LOADING & CLEANING

This section loads the dataset, performs initial cleaning, and prepares the data for analysis. Only creator accounts are retained to ensure relevance for influencer marketing analysis. Date-time fields are processed, and missing values are handled.

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Instagram_Analytics.csv to Instagram_Analytics.csv


In [5]:
# Load dataset
df = pd.read_csv('/content/Instagram_Analytics.csv')

# Keeping only creators
df = df[df['account_type'] == 'creator'].copy()

df = df.drop(columns=['account_type'])

# Convert to datetime
df['post_datetime'] = pd.to_datetime(df['post_datetime'], errors='coerce')

# Extract date and time
df['post_date'] = df['post_datetime'].dt.date
df['post_time'] = df['post_datetime'].dt.strftime('%H:%M')

# Drop missing critical values
df = df.dropna(subset=['likes','comments','shares','saves','reach','follower_count'])

# Reset index
df = df.reset_index(drop=True)

df.shape

(20944, 23)

### Data Preparation Summary

The dataset was filtered to include only creator accounts to maintain analytical consistency. Datetime fields were standardized, missing critical engagement values were removed, and posting timestamps were separated into date and time components for temporal analysis.

These preprocessing steps ensure reliable feature engineering and downstream statistical modeling.

## SECTION 2 - OUTLIER FLAGGING

Extreme engagement values can disproportionately affect statistical analysis and model behavior. To identify unusually high-performing posts, outliers were detected using the 3-sigma rule based on engagement distributions.

These observations help identify potentially viral or anomalous content patterns.

In [6]:
# OUTLIER FLAGGING (3σ rule)
for col in ["likes", "comments", "shares", "saves"]:
    upper = df[col].mean() + 3 * df[col].std()
    df[f"{col}_outlier"] = (df[col] > upper).astype(int)

print("Outlier counts per metric:")
print(df[["likes_outlier","comments_outlier","shares_outlier","saves_outlier"]].sum())

Outlier counts per metric:
likes_outlier       343
comments_outlier    364
shares_outlier      341
saves_outlier       336
dtype: int64


##SECTION 3 — FEATURE ENGINEERING

To better understand audience interaction quality, multiple engagement intelligence features were engineered from raw Instagram metrics.

The project categorizes engagement into:

- **Affective Engagement** → likes
- **Cognitive Engagement** → comments
- **Behavioral Engagement** → shares and saves

Additional normalized metrics such as virality score, save efficiency, and engagement rates were developed to compare creators independent of audience size.

A weighted engagement depth score was also introduced to prioritize higher-value interactions such as shares and saves over passive likes.

In [7]:
# FEATURE ENGINEERING
# Core reach-normalised ratios (formulas unchanged)
df["er_recomputed"]   = (df["likes"] + df["comments"] + df["shares"] + df["saves"])                         / df["reach"].replace(0, np.nan)
df["virality_score"]  = df["shares"] / df["reach"].replace(0, np.nan)
df["saves_to_reach"]  = df["saves"]  / df["reach"].replace(0, np.nan)
df["shares_to_reach"] = df["shares"] / df["reach"].replace(0, np.nan)
df["lc_ratio"]        = df["likes"]  / (df["comments"] + 1)
df["comment_density"] = df["comments"] / (df["likes"] + 1)

# [ADDED] Three engagement dimension scores (per-reach normalised)
# Maps the cognitive–affective–behavioral framework to platform metrics:
#   affective  → likes       : passive emotional reaction
#   cognitive  → comments    : active dialogue and thinking
#   behavioral → shares+saves: deliberate redistribution or bookmark (high intent)
df["affective_score"]  = df["likes"]                    / df["reach"].replace(0, np.nan)
df["cognitive_score"]  = df["comments"]                 / df["reach"].replace(0, np.nan)
df["behavioral_score"] = (df["shares"] + df["saves"])   / df["reach"].replace(0, np.nan)

# [ADDED] depth_score — cognitive engagement depth proxy
# Clip at 99th pct then log1p to suppress the extreme tail (raw max = 84)
_clip = df["comment_density"].quantile(0.99)
df["comment_density_log"] = np.log1p(df["comment_density"].clip(upper=_clip))
df["depth_score"]         = df["comment_density_log"]   # named alias

# [ADDED] behavioral_rate — proportion of high-intent actions vs all actions
# (shares + saves) / (likes + comments + 1)
# Higher = audience moving beyond passive scrolling into intentional behavior
df["behavioral_rate"] = (df["shares"] + df["saves"]) / (df["likes"] + df["comments"] + 1)

# [ADDED] effectiveness_score (EQI) — normalised weighted composite
# Behavioral: 0.30×ER + 0.20×virality  |  Affective: 0.30×saves  |  Cognitive: 0.20×depth
_eqi_cols = ["er_recomputed", "virality_score", "saves_to_reach", "comment_density_log"]
_scaler   = MinMaxScaler()
_normed   = _scaler.fit_transform(df[_eqi_cols].fillna(0))
_ndf      = pd.DataFrame(_normed, columns=[c+"_norm" for c in _eqi_cols], index=df.index)
df["effectiveness_score"] = (
    0.30 * _ndf["er_recomputed_norm"]      +
    0.20 * _ndf["virality_score_norm"]     +
    0.30 * _ndf["saves_to_reach_norm"]     +
    0.20 * _ndf["comment_density_log_norm"]
)

print("\nFeature summary (creator posts only):")
print(df[["er_recomputed","virality_score","saves_to_reach",
          "depth_score","behavioral_rate","effectiveness_score"]].describe().round(4))


Feature summary (creator posts only):
       er_recomputed  virality_score  saves_to_reach  depth_score  behavioral_rate  effectiveness_score
count     20944.0000      20944.0000      20944.0000   20944.0000       20944.0000           20944.0000
mean          0.0562          0.0023          0.0068       0.0577           0.3317               0.1225
std           0.0312          0.0015          0.0042       0.2256           2.0905               0.0678
min           0.0000          0.0000          0.0000       0.0000           0.0000               0.0000
25%           0.0325          0.0012          0.0037       0.0200           0.1690               0.0717
50%           0.0547          0.0021          0.0065       0.0284           0.1917               0.1172
75%           0.0769          0.0032          0.0093       0.0367           0.2128               0.1644
max           0.2998          0.0184          0.0574       1.9459         159.0000               0.7168


- Recomputed engagement rate as (likes + comments + shares + saves) / reach
- Derived virality score, saves-to-reach, and shares-to-reach as reach-normalised ratios
- Constructed affective, cognitive, and behavioral dimension scores mapped to likes, comments, and (shares + saves) respectively
- Applied log-transform and 99th percentile clip to comment density to reduce skew
- Built behavioral_rate as the share of total engagement driven by saves and shares
- Computed Engagement Quality Index (EQI / effectiveness_score) as a min-max normalised weighted composite of ER, virality, saves-to-reach, and comment density

## SECTION 4 - CREATOR SEGMENTATION

### Why Creator Segmentation Matters

Influencers operate at different audience scales, which significantly impacts engagement behavior, audience trust, and reach dynamics. Engagement rates and interaction patterns often vary substantially between smaller niche creators and larger high-reach accounts.

To account for these differences, creators were segmented into follower-based tiers. This allows engagement patterns and performance metrics to be compared more fairly across creator categories.

Traditional global influencer classifications typically define creators as:
- Nano: <10K followers
- Micro: 10K–100K followers
- Macro: 100K+ followers

However, the dataset used in this project primarily contains nano and micro influencers, with follower counts ranging approximately between 3K and 31K. Applying standard global thresholds directly would result in highly imbalanced groups and weak analytical contrast.

Therefore, adjusted thresholds were created to preserve meaningful segmentation within the dataset while remaining aligned with influencer marketing principles and Indian creator-market dynamics.

| Creator Tier | Follower Range | Characteristics |
|---|---|---|
| Nano | <7,500 | Hyper-local creators with highly authentic engagement |
| Micro | 7,500–15,000 | Niche-engaged creators commonly used in D2C and targeted campaigns |
| Macro | >15,000 | Higher-reach creators with broader audience visibility within this dataset |

This segmentation strategy improves:
- interpretability of engagement patterns,
- comparison across creator categories,
- and realism of influencer marketing analysis.

The resulting tier distribution remained reasonably balanced, enabling stronger comparative analysis across creator groups.

In [8]:
# FOLLOWER TIER
df["creator_tier"] = pd.cut(
    df["follower_count"],
    bins=[0, 7_500, 15_000, float("inf")],
    labels=["nano", "micro", "macro"]
)

print("Creator tier distribution:")
_ts = df.groupby("creator_tier", observed=True)["follower_count"]         .agg(["count", "min", "mean", "max"]).round(0)
_ts.columns = ["n_posts", "min_followers", "avg_followers", "max_followers"]
print(_ts)

# TIME BINS (unchanged)
df["time_bin"] = pd.cut(
    df["post_hour"],
    bins=[0, 6, 12, 17, 21, 24],
    labels=["night", "morning", "afternoon", "evening", "late_night"],
    right=False
)

# ENCODING
df["media_type_enc"]   = df["media_type"].map({"image": 0, "reel": 1, "carousel": 2})

# NORMALIZATION (unchanged cols)
_norm_cols = ["er_recomputed", "virality_score", "saves_to_reach", "comment_density"]
df[[c + "_norm" for c in _norm_cols]] = MinMaxScaler().fit_transform(df[_norm_cols])

Creator tier distribution:
              n_posts  min_followers  avg_followers  max_followers
creator_tier                                                      
nano             8360           3083         4993.0           7486
micro            8405           8167        10108.0          13798
macro            4179          15923        21298.0          31095


## SECTION 5 - ENGAGEMENT ANALYSIS
#### Affective Engagement: Caption Depth
- Binned caption length into short / medium / long using tertile split
- Cross-tabulated caption depth against performance buckets
- Computed Pearson correlations between caption length and four engagement metrics

In [9]:
# AFFECTIVE ENGAGEMENT
df["caption_depth"] = pd.qcut(
    df["caption_length"],
    q=3,
    labels=["short", "medium", "long"]
)

ct = pd.crosstab(
    df["caption_depth"],
    df["performance_bucket_label"],
    normalize="index"
)

print("\nCaption depth vs performance:")
print(ct.round(3))

# CORRELATION CHECK
corr_targets = [
    "er_recomputed",
    "virality_score",
    "saves_to_reach",
    "comment_density"
]

print("\nCaption length correlations:")
print(
    df[["caption_length"] + corr_targets]
    .corr()["caption_length"]
    .drop("caption_length")
    .round(4)
)


Caption depth vs performance:
performance_bucket_label   high    low  medium  viral
caption_depth                                        
short                     0.250  0.252   0.249  0.250
medium                    0.253  0.246   0.252  0.250
long                      0.252  0.253   0.245  0.249

Caption length correlations:
er_recomputed      0.0017
virality_score     0.0039
saves_to_reach     0.0015
comment_density    0.0055
Name: caption_length, dtype: float64
